# IQOTH Lung Cancer CT — Preprocessing & EDA

**Self-contained notebook** — no local imports required. Runs on Kaggle as-is.

**Pipeline**
1. Configuration
2. Dataset inspection
3. Sample images per class
4. Pixel intensity statistics
5. Patient-level split (no leakage)
6. Class imbalance & weights
7. Preprocessing transform visualization
8. Slices-per-case distribution
9. Export `splits.csv`
10. Final summary

---
## 0 · Imports & Configuration

> ⚙️ **Edit `DATASET_ROOT` to match your Kaggle dataset path.**

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os, re, csv, random
from pathlib import Path
from collections import Counter, defaultdict

# ── Third-party ──────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torchvision.transforms as T

plt.style.use("seaborn-v0_8-whitegrid")

# ════════════════════════════════════════════════════════════════════════════
# CONFIG — edit these paths for your environment
# ════════════════════════════════════════════════════════════════════════════

DATASET_ROOT = "/home/metal/Experiment/data/The IQ-OTHNCCD lung cancer dataset"
# On Kaggle it will be something like:
# DATASET_ROOT = "/kaggle/input/iq-othnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset"

SPLITS_CSV   = Path("./splits.csv")   # output location

# Dataset
CLASS_FOLDERS  = ["Benign cases", "Malignant cases", "Normal cases"]
CLASS_TO_LABEL = {"Benign cases": 0, "Malignant cases": 1, "Normal cases": 2}
LABEL_TO_CLASS = {0: "Benign", 1: "Malignant", 2: "Normal"}
NUM_CLASSES    = 3

# Split ratios (applied at the patient/case level)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
SPLIT_SEED  = 42

# Image
IMG_SIZE     = 224     # resize target
IMG_CHANNELS = 1       # grayscale

# Colour palette for plots
COLORS = {"Benign": "steelblue", "Malignant": "firebrick", "Normal": "seagreen"}

random.seed(SPLIT_SEED); np.random.seed(SPLIT_SEED)
print(f"Dataset root : {DATASET_ROOT}")
print(f"Splits CSV   : {SPLITS_CSV.resolve()}")

---
## 0b · Helper Functions

All functions are defined here so the notebook is self-contained.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Dataset utilities
# ════════════════════════════════════════════════════════════════════════════

def extract_case_id(filename: str) -> int:
    """
    Extract the numeric patient/case ID from a filename.
    'Benign case (7).jpg'  → 7
    Fallback: stable hash of the filename.
    """
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else abs(hash(filename)) % 100_000


def build_case_registry(dataset_root: str = DATASET_ROOT) -> list:
    """
    Walk DATASET_ROOT and return a list of records:
      { image_path, class_name, label, case_id, filename }
    """
    root = Path(dataset_root)
    records = []
    for cls_folder in CLASS_FOLDERS:
        cls_dir = root / cls_folder
        if not cls_dir.exists():
            print(f"⚠️  Folder not found: {cls_dir}")
            continue
        label = CLASS_TO_LABEL[cls_folder]
        for img_file in sorted(cls_dir.iterdir()):
            if img_file.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                records.append({
                    'image_path': str(img_file),
                    'class_name': cls_folder,
                    'label'     : label,
                    'case_id'   : extract_case_id(img_file.name),
                    'filename'  : img_file.name,
                })
    return records


def patient_level_split(records: list, seed: int = SPLIT_SEED) -> list:
    """
    Stratified, patient-level train/val/test split.
    All slices from the same case land in the same split → no leakage.
    Returns records extended with a 'split' key.
    """
    rng = random.Random(seed)
    class_cases = defaultdict(set)
    for r in records:
        class_cases[r['class_name']].add(r['case_id'])

    split_map = {}
    for cls, case_ids in class_cases.items():
        ids = sorted(case_ids)
        rng.shuffle(ids)
        n = len(ids)
        n_train = max(1, round(n * TRAIN_RATIO))
        n_val   = max(1, round(n * VAL_RATIO))
        for c in ids[:n_train]:              split_map[c] = 'train'
        for c in ids[n_train:n_train+n_val]: split_map[c] = 'val'
        for c in ids[n_train+n_val:]:        split_map[c] = 'test'
        print(f"  {cls}: {n_train} train / {n_val} val / {n - n_train - n_val} test cases")

    out = []
    for r in records:
        r2 = dict(r)
        r2['split'] = split_map.get(r['case_id'], 'train')
        out.append(r2)
    return out


def verify_no_leakage(records: list):
    """Assert every patient case appears in exactly one split."""
    case_splits = defaultdict(set)
    for r in records:
        case_splits[r['case_id']].add(r['split'])
    leaks = {c: s for c, s in case_splits.items() if len(s) > 1}
    if leaks:
        raise ValueError(f"❌ Patient leakage! Cases in multiple splits: {leaks}")
    print("✅ No patient leakage — each case appears in exactly one split.")


def get_class_weights(records: list) -> torch.Tensor:
    """Inverse-frequency class weights for CrossEntropyLoss."""
    train_recs = [r for r in records if r['split'] == 'train']
    counts = Counter(r['label'] for r in train_recs)
    total  = sum(counts.values())
    w = torch.zeros(NUM_CLASSES)
    for lbl, cnt in counts.items():
        w[lbl] = total / (NUM_CLASSES * cnt)
    return w


def save_splits_csv(records: list, csv_path=SPLITS_CSV):
    """Persist splits to a CSV file."""
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    fields = ['image_path', 'class_name', 'label', 'case_id', 'filename', 'split']
    with open(csv_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader(); w.writerows(records)
    print(f"✅ splits.csv → {csv_path}  ({len(records)} rows)")


# ════════════════════════════════════════════════════════════════════════════
# Transform helpers
# ════════════════════════════════════════════════════════════════════════════

def get_transforms(split: str, augment: bool = False, img_size: int = IMG_SIZE):
    """
    Returns a torchvision Compose pipeline.
    - val/test : resize → grayscale → to-tensor → normalize
    - train+aug: adds flip, rotation, affine, colour-jitter before to-tensor
    """
    mean, std = [0.5], [0.5]
    base = [T.Resize((img_size, img_size)), T.Grayscale(num_output_channels=1)]
    aug  = [
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        T.ColorJitter(brightness=0.15, contrast=0.15),
    ] if (split == 'train' and augment) else []
    end  = [T.ToTensor(), T.Normalize(mean=mean, std=std)]
    return T.Compose(base + aug + end)


def tensor_to_np(t):
    """Convert a [-1,1] tensor (1,H,W) to a uint8 numpy array (H,W)."""
    arr = t.squeeze(0).numpy()
    arr = (arr + 1.0) / 2.0          # [-1,1] → [0,1]
    return (arr * 255).clip(0, 255).astype(np.uint8)


print("✅ All helper functions defined.")

---
## 1 · Dataset Inspection

In [ ]:
records = build_case_registry()
total_cases = len(set(r['case_id'] for r in records))
print(f"Total images : {len(records)}")
print(f"Total cases  : {total_cases}")
print()

by_class = defaultdict(list)
for r in records:
    by_class[r['class_name']].append(r)

print(f"{'Class':<20} {'Cases':>6} {'Slices':>8} {'Slices/Case':>12}")
print("-" * 50)
for cls in CLASS_FOLDERS:
    rows    = by_class[cls]
    n_cases = len(set(r['case_id'] for r in rows))
    print(f"{cls:<20} {n_cases:>6} {len(rows):>8} {len(rows)/n_cases:>12.1f}")

In [ ]:
# Image size distribution (sampled)
sample_recs = random.sample(records, min(200, len(records)))
sizes = [Image.open(r['image_path']).size for r in sample_recs]  # (W, H)
widths, heights = zip(*sizes)

print(f"Width  — min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}")
print(f"Height — min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(widths,  bins=20, color='steelblue', edgecolor='white')
axes[0].set_title("Image Widths"); axes[0].set_xlabel("pixels")
axes[1].hist(heights, bins=20, color='darkorange', edgecolor='white')
axes[1].set_title("Image Heights"); axes[1].set_xlabel("pixels")
plt.suptitle("Raw Image Size Distribution (sample=200)", fontsize=12)
plt.tight_layout()
plt.savefig("image_size_distribution.png", dpi=120, bbox_inches='tight')
plt.show()

---
## 2 · Sample Images per Class

In [ ]:
N_SAMPLES = 5
fig, axes = plt.subplots(len(CLASS_FOLDERS), N_SAMPLES, figsize=(14, 8))

for row_i, cls in enumerate(CLASS_FOLDERS):
    samples = random.sample(by_class[cls], min(N_SAMPLES, len(by_class[cls])))
    for col_i, r in enumerate(samples):
        img = Image.open(r['image_path']).convert('L')
        axes[row_i, col_i].imshow(img, cmap='gray')
        axes[row_i, col_i].axis('off')
        axes[row_i, col_i].set_title(r['filename'][:18], fontsize=7)
    axes[row_i, 0].set_ylabel(cls.split(' ')[0], fontsize=12, labelpad=8)

plt.suptitle("Sample CT Slices per Class (Grayscale)", fontsize=13)
plt.tight_layout()
plt.savefig("sample_images.png", dpi=120, bbox_inches='tight')
plt.show()

---
## 3 · Pixel Intensity Statistics

In [ ]:
SAMPLE_PER_CLASS = 100
stats = {}

for cls in CLASS_FOLDERS:
    subset  = random.sample(by_class[cls], min(SAMPLE_PER_CLASS, len(by_class[cls])))
    all_px  = np.concatenate([
        np.array(Image.open(r['image_path']).convert('L'), dtype=np.float32).flatten() / 255.0
        for r in subset
    ])
    stats[cls] = {'mean': all_px.mean(), 'std': all_px.std()}
    print(f"{cls:<20}  mean={stats[cls]['mean']:.4f}  std={stats[cls]['std']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, cls in zip(axes, CLASS_FOLDERS):
    short = cls.split(' ')[0]
    subset = random.sample(by_class[cls], min(SAMPLE_PER_CLASS, len(by_class[cls])))
    px = np.concatenate([
        np.array(Image.open(r['image_path']).convert('L'), dtype=np.float32).flatten() / 255.0
        for r in subset
    ])
    ax.hist(px, bins=64, color=COLORS[short], alpha=0.8, edgecolor='none')
    ax.set_title(f"{short}\nmean={stats[cls]['mean']:.3f}, std={stats[cls]['std']:.3f}")
    ax.set_xlabel("Pixel value (0–1)")
plt.suptitle("Pixel Intensity Distribution per Class", fontsize=12)
plt.tight_layout()
plt.savefig("pixel_intensity_distribution.png", dpi=120, bbox_inches='tight')
plt.show()

---
## 4 · Patient-Level Split (No Leakage)

In [ ]:
split_records = patient_level_split(records)
verify_no_leakage(split_records)
print()

print(f"{'Split':<8} {'Cases':>6} {'Slices':>8}  Class distribution")
print("-" * 60)
for split in ['train', 'val', 'test']:
    sub     = [r for r in split_records if r['split'] == split]
    n_cases = len(set(r['case_id'] for r in sub))
    cnt     = Counter(r['class_name'].split(' ')[0] for r in sub)
    print(f"{split:<8} {n_cases:>6} {len(sub):>8}  {dict(cnt)}")

In [ ]:
# Stacked bar: slice count per class per split
splits     = ['train', 'val', 'test']
class_keys = [c.split(' ')[0] for c in CLASS_FOLDERS]
data = {
    split: [Counter(r['class_name'].split(' ')[0] for r in split_records if r['split']==split).get(k, 0)
            for k in class_keys]
    for split in splits
}

x, bottom = np.arange(len(splits)), np.zeros(len(splits))
fig, ax = plt.subplots(figsize=(7, 4))
for i, cls in enumerate(class_keys):
    vals = [data[s][i] for s in splits]
    bars = ax.bar(x, vals, bottom=bottom, label=cls, color=COLORS[cls], alpha=0.85)
    for rect, v in zip(bars, vals):
        if v > 0:
            ax.text(rect.get_x() + rect.get_width()/2, rect.get_y() + rect.get_height()/2,
                    str(v), ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    bottom += np.array(vals)

ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_ylabel("CT slices"); ax.set_title("Class Distribution — Patient-Level Splits")
ax.legend(); plt.tight_layout()
plt.savefig("split_class_distribution.png", dpi=120, bbox_inches='tight')
plt.show()

---
## 5 · Class Imbalance & Weights

In [ ]:
weights = get_class_weights(split_records)
print("Inverse-frequency class weights (for CrossEntropyLoss):")
for i, w in enumerate(weights):
    print(f"  {LABEL_TO_CLASS[i]:<12}: {w:.4f}")

train_sub = [r for r in split_records if r['split'] == 'train']
cnt_train = Counter(r['class_name'].split(' ')[0] for r in train_sub)

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    cnt_train.values(), labels=cnt_train.keys(),
    autopct='%1.1f%%', colors=[COLORS[k] for k in cnt_train],
    startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
ax.set_title("Train Set — Class Proportion")
plt.tight_layout()
plt.savefig("train_class_proportion.png", dpi=120, bbox_inches='tight')
plt.show()

---
## 6 · Preprocessing Transform Visualization

Each row: **Raw → No-aug (×3) → Augmented (×3)**

In [ ]:
tf_noop = get_transforms('val',   augment=False)
tf_aug  = get_transforms('train', augment=True)
N_SHOW  = 3

fig, axes = plt.subplots(len(CLASS_FOLDERS), 1 + N_SHOW * 2, figsize=(16, 9))

for row_i, cls in enumerate(CLASS_FOLDERS):
    r       = random.choice(by_class[cls])
    raw_img = Image.open(r['image_path']).convert('L')

    # Col 0: raw
    axes[row_i, 0].imshow(raw_img, cmap='gray')
    axes[row_i, 0].axis('off')
    axes[row_i, 0].set_ylabel(cls.split(' ')[0], fontsize=11)
    if row_i == 0: axes[row_i, 0].set_title("Raw", fontsize=9)

    # Cols 1–N_SHOW: no augmentation
    for j in range(N_SHOW):
        axes[row_i, 1+j].imshow(tensor_to_np(tf_noop(raw_img)), cmap='gray')
        axes[row_i, 1+j].axis('off')
        if row_i == 0: axes[row_i, 1+j].set_title("No-aug", fontsize=9)

    # Cols N_SHOW+1–end: augmented
    for j in range(N_SHOW):
        axes[row_i, 1+N_SHOW+j].imshow(tensor_to_np(tf_aug(raw_img)), cmap='gray')
        axes[row_i, 1+N_SHOW+j].axis('off')
        if row_i == 0: axes[row_i, 1+N_SHOW+j].set_title(f"Aug #{j+1}", fontsize=9)

plt.suptitle(f"Transforms  (resize→{IMG_SIZE}×{IMG_SIZE}, grayscale, norm [-1,1])", fontsize=12)
plt.tight_layout()
plt.savefig("transform_visualization.png", dpi=120, bbox_inches='tight')
plt.show()

---
## 7 · Slices per Patient Case

In [ ]:
case_slices = defaultdict(lambda: defaultdict(int))
for r in records:
    case_slices[r['class_name'].split(' ')[0]][r['case_id']] += 1

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, cls in zip(axes, [c.split(' ')[0] for c in CLASS_FOLDERS]):
    counts = list(case_slices[cls].values())
    ax.hist(counts, bins=20, color=COLORS[cls], edgecolor='white', alpha=0.85)
    ax.set_title(f"{cls}\nmin={min(counts)}, max={max(counts)}, mean={np.mean(counts):.0f}")
    ax.set_xlabel("Slices per case")
plt.suptitle("Slices per Patient Case Distribution", fontsize=12)
plt.tight_layout()
plt.savefig("slices_per_case.png", dpi=120, bbox_inches='tight')
plt.show()

---
## 8 · Export `splits.csv`

In [ ]:
save_splits_csv(split_records, SPLITS_CSV)

# Quick read-back check
with open(SPLITS_CSV) as f:
    loaded = list(csv.DictReader(f))
print(f"Loaded back {len(loaded)} rows — first row:")
print({k: loaded[0][k] for k in ['class_name', 'case_id', 'split']})

---
## 9 · Final Summary

In [ ]:
print("=" * 58)
print("  IQOTH Dataset Preprocessing — Summary")
print("=" * 58)
print(f"  Total images        : {len(records)}")
print(f"  Total patient cases : {len(set(r['case_id'] for r in records))}")
print(f"  Image size (model)  : {IMG_SIZE}×{IMG_SIZE} grayscale")
print(f"  Normalization       : mean=0.5, std=0.5  → [-1, 1]")
print(f"  Split strategy      : Patient-level (stratified, no leakage)")
print(f"  Train/Val/Test      : {TRAIN_RATIO:.0%} / {VAL_RATIO:.0%} / {TEST_RATIO:.0%}")
print(f"  Splits CSV          : {SPLITS_CSV.resolve()}")
print()
for split in ['train', 'val', 'test']:
    sub     = [r for r in split_records if r['split'] == split]
    n_cases = len(set(r['case_id'] for r in sub))
    cnt     = Counter(r['class_name'].split(' ')[0] for r in sub)
    print(f"  {split:<6}: {len(sub):>5} slices | {n_cases:>3} cases | {dict(cnt)}")
print("=" * 58)
print("✅ Done — splits.csv is ready for all downstream experiments.")